In [ ]:
#creating spark session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Retail Data Engineering Project") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [ ]:
#importing python libraries
import pandas as pd
import numpy as np


In [ ]:
#creation of  customer table with required rows and columns simultaneously  converting to data frame!
customers = pd.DataFrame({
    "customer_id": range(1,100001),
    "customer_name": [f"Customer_{i}" for i in range(1,100001)],
    "state": np.random.choice(
        ["Karnataka","Tamil Nadu","Telangana","Maharashtra","Kerala"],
        100000
    ),
    "city": np.random.choice(
        ["Bangalore","Chennai","Hyderabad","Mumbai","Kochi"],
        100000
    )
})

customers.head()

,customer_id,customer_name,state,city
0,1,Customer_1,Telangana,Kochi
1,2,Customer_2,Karnataka,Bangalore
2,3,Customer_3,Telangana,Hyderabad
3,4,Customer_4,Kerala,Hyderabad
4,5,Customer_5,Karnataka,Bangalore


In [ ]:
#creation of  products table with required rows and columns simultaneously  converting to data frame!
products = pd.DataFrame({
    "product_id": range(1,50001),
    "product_name": [f"Product_{i}" for i in range(1,50001)],
    "category": np.random.choice(
        ["Electronics","Fashion","Books","Home","Sports"],
        50000
    ),
    "price": np.random.randint(100,10000,50000)
})

products.head()

,product_id,product_name,category,price
0,1,Product_1,Books,4857
1,2,Product_2,Electronics,6290
2,3,Product_3,Fashion,2898
3,4,Product_4,Home,3529
4,5,Product_5,Books,4008


In [ ]:
#creation of  orders table with required rows and columns simultaneously  converting to data frame!
orders = pd.DataFrame({
    "order_id": range(1,1000001),
    "customer_id": np.random.randint(1,100001,1000000),
    "product_id": np.random.randint(1,50001,1000000),
    "quantity": np.random.randint(1,10,1000000),
    "order_amount": np.random.randint(100,10000,1000000)
})

orders.head()

,order_id,customer_id,product_id,quantity,order_amount
0,1,9339,34047,2,9535
1,2,8617,2890,2,5199
2,3,40436,3537,7,5587
3,4,74037,44118,7,766
4,5,12225,30797,7,3518


In [ ]:
#finding number of rows using len function in python
print(len(customers))
print(len(products))
print(len(orders))

100000
50000
1000000


In [ ]:
#shape is used to find the schema of particular column
products.shape


(50000, 4)

In [ ]:
customers.to_csv("customers.csv", index=False)

products.to_csv("products.csv", index=False)

orders.to_csv("orders.csv", index=False)

print("CSV files saved successfully")

CSV files saved successfully


In [ ]:
!pwd

/content


In [ ]:
!find /content/sample_data -name "*.csv"

/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_test.csv


In [ ]:
customers.to_csv(
    "/content/drive/MyDrive/customers.csv",
    index=False
)
products.to_csv(
    "/content/drive/MyDrive/products.csv",
    index=False
)
orders.to_csv(
    "/content/drive/MyDrive/orders.csv",
    index=False
)

In [ ]:
customers_df = spark.read.csv(
    "/content/drive/MyDrive/customers.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "/content/drive/MyDrive/products.csv",
    header=True,
    inferSchema=True
)

orders_df = spark.read.csv(
    "/content/drive/MyDrive/orders.csv",
    header=True,
    inferSchema=True
)

### Bronze Layer: Saving Raw Data to Parquet

It's a best practice to create a 'Bronze' layer where raw, ingested data is stored in a performant format like Parquet. This layer serves as an immutable record of the source data, allowing for re-processing the data if issues are found further downstream or if business rules change. Here, we'll save the DataFrames loaded from CSVs (`customers_df`, `products_df`, `orders_df`) into a Bronze directory.

In [ ]:
import os

# Define the base path for the Bronze layer in Google Drive
bronze_base_path = "/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze"

# Create directories for customers, products, and orders if they don't exist
for folder in ["customers", "products", "orders"]:
    path = os.path.join(bronze_base_path, folder)
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"Created Bronze directory: {path}")
    else:
        print(f"Bronze directory already exists: {path}")


Bronze directory already exists: /content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze/customers
Bronze directory already exists: /content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze/products
Bronze directory already exists: /content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze/orders


In [ ]:
# Save the initial DataFrames to the Bronze layer as Parquet files
customers_df.write.mode("overwrite").parquet(
    os.path.join(bronze_base_path, "customers")
)
print("customers_df saved to Bronze layer.")

products_df.write.mode("overwrite").parquet(
    os.path.join(bronze_base_path, "products")
)
print("products_df saved to Bronze layer.")

orders_df.write.mode("overwrite").parquet(
    os.path.join(bronze_base_path, "orders")
)
print("orders_df saved to Bronze layer.")

customers_df saved to Bronze layer.
products_df saved to Bronze layer.
orders_df saved to Bronze layer.


### Silver Layer: Reading from Bronze for Transformations

With the Bronze layer established, the Silver layer will now read its input from the Bronze Parquet files. This ensures that all Silver layer transformations, quality checks, and enrichments are performed on the standardized and persistent raw data from the Bronze layer.

In [ ]:
import os

# Define the base path for the Bronze layer
bronze_base_path = "/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze"

# Read the Bronze layer parquet files into new DataFrames
customers_bronze_df = spark.read.parquet(
    os.path.join(bronze_base_path, "customers")
)
products_bronze_df = spark.read.parquet(
    os.path.join(bronze_base_path, "products")
)
orders_bronze_df = spark.read.parquet(
    os.path.join(bronze_base_path, "orders")
)

print("Bronze DataFrames loaded successfully for Silver layer processing.")

Bronze DataFrames loaded successfully for Silver layer processing.


Now that the raw data is safely stored in the Bronze layer, any subsequent transformations for the Silver layer should ideally read from these Bronze Parquet files rather than directly from the original CSVs. This modular approach enhances reliability and maintainability of your data pipeline.

In [ ]:
print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("Orders:", orders_df.count())

Customers: 100000
Products: 50000
Orders: 1000000


In [ ]:
print(products_df.inputFiles())

['file:///content/drive/MyDrive/products.csv']


In [ ]:
products_df.count()

50000

In [ ]:
import pandas as pd

prod_check = pd.read_csv("/content/drive/MyDrive/products.csv")

print(prod_check.shape)

(50000, 4)


In [ ]:

prod_check.columns

Index(['product_id', 'product_name', 'category', 'price'], dtype='object')

In [ ]:
products.to_csv(
    "/content/drive/MyDrive/products.csv",
    index=False
)

In [ ]:
prod_check = pd.read_csv(
    "/content/drive/MyDrive/products.csv"
)

print(prod_check.shape)
print(prod_check.columns)

(50000, 4)
Index(['product_id', 'product_name', 'category', 'price'], dtype='object')


In [ ]:
#As part of the Silver layer, I validated the schema using printSchema() to ensure that the imported data matched the expected data types.
customers_df.printSchema()

products_df.printSchema()

orders_df.printSchema()
print("Schema Validation")

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_amount: integer (nullable = true)

Schema Validation


In [ ]:
from pyspark.sql.functions import col

def null_check(df):
    for c in df.columns:
        print(c, df.filter(col(c).isNull()).count())

In [ ]:
null_check(customers_df)

null_check(products_df)

null_check(orders_df)

customer_id 0
customer_name 0
state 0
city 0
product_id 0
product_name 0
category 0
price 0
order_id 0
customer_id 0
product_id 0
quantity 0
order_amount 0


In [ ]:
from pyspark.sql.functions import col

# Re-create Customers_silver (deduplication and null checks)
Customers_silver = customers_bronze_df.dropDuplicates(["customer_id"])
Customers_silver = Customers_silver.filter(col("customer_id").isNotNull())

Customers_silver.groupBy("customer_id") \
            .count() \
            .filter(col("count") > 1) \
            .count()

0

### Troubleshooting Py4JJavaError: File Not Found

The `Py4JJavaError` you encountered indicates that Spark was unable to locate a specific Parquet file (`part-00002-....snappy.parquet`) when trying to access the `Customers_silver` DataFrame. This usually means that the underlying data in the Bronze layer for customers is either incomplete or has been corrupted.

To fix this, we need to re-run the steps that create and save the customer data to the Bronze layer, and then re-derive the `Customers_silver` DataFrame. This will ensure that all necessary Parquet files are present and accessible.

In [ ]:
# Step 1: Re-save the initial pandas DataFrame to CSV (Ensuring fresh source data)
customers.to_csv(
    "/content/drive/MyDrive/customers.csv",
    index=False
)
products.to_csv(
    "/content/drive/MyDrive/products.csv",
    index=False
)
orders.to_csv(
    "/content/drive/MyDrive/orders.csv",
    index=False
)
print("Original CSV files re-saved to Google Drive.")

Original CSV files re-saved to Google Drive.


In [ ]:
# Step 2: Re-load the CSV files into Spark DataFrames
customers_df = spark.read.csv(
    "/content/drive/MyDrive/customers.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "/content/drive/MyDrive/products.csv",
    header=True,
    inferSchema=True
)

orders_df = spark.read.csv(
    "/content/drive/MyDrive/orders.csv",
    header=True,
    inferSchema=True
)
print("Spark DataFrames re-loaded from CSV.")

Spark DataFrames re-loaded from CSV.


In [ ]:
import os

# Define the base path for the Bronze layer in Google Drive
bronze_base_path = "/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze"

# Step 3: Re-save the Spark DataFrames to the Bronze layer as Parquet files (overwriting any incomplete data)
customers_df.write.mode("overwrite").parquet(
    os.path.join(bronze_base_path, "customers")
)
print("customers_df re-saved to Bronze layer.")

products_df.write.mode("overwrite").parquet(
    os.path.join(bronze_base_path, "products")
)
print("products_df re-saved to Bronze layer.")

orders_df.write.mode("overwrite").parquet(
    os.path.join(bronze_base_path, "orders")
)
print("orders_df re-saved to Bronze layer.")

customers_df re-saved to Bronze layer.
products_df re-saved to Bronze layer.
orders_df re-saved to Bronze layer.


In [ ]:
import os

# Define the base path for the Bronze layer
bronze_base_path = "/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze"

# Step 4: Re-read the Bronze layer parquet files into new DataFrames
customers_bronze_df = spark.read.parquet(
    os.path.join(bronze_base_path, "customers")
)
products_bronze_df = spark.read.parquet(
    os.path.join(bronze_base_path, "products")
)
orders_bronze_df = spark.read.parquet(
    os.path.join(bronze_base_path, "orders")
)
print("Bronze DataFrames re-loaded successfully for Silver layer processing.")

Bronze DataFrames re-loaded successfully for Silver layer processing.


In [ ]:
from pyspark.sql.functions import col

# Step 5: Re-create Customers_silver (deduplication and null checks)
Customers_silver = customers_bronze_df.dropDuplicates(["customer_id"])
Customers_silver = Customers_silver.filter(col("customer_id").isNotNull())

products_silver = products_bronze_df.dropDuplicates(["product_id"])
products_silver = products_silver.filter(col("product_id").isNotNull())
products_silver = products_silver.filter(col("price")>0)

orders_silver = orders_bronze_df.dropDuplicates(["order_id"])
orders_silver = orders_silver.filter(col("order_id").isNotNull())
orders_silver = orders_silver.filter(col("quantity")>0)
orders_silver = orders_silver.filter(col("order_amount")>0)

print("Customers_silver, products_silver, and orders_silver DataFrames re-created with quality checks.")

Customers_silver, products_silver, and orders_silver DataFrames re-created with quality checks.


In [ ]:
# Step 6: Now, re-run the duplicate check on Customers_silver
duplicate_customer_ids = Customers_silver.groupBy("customer_id") \
            .count() \
            .filter(col("count") > 1) \
            .count()

print(f"Number of duplicate customer_ids in Customers_silver: {duplicate_customer_ids}")

Number of duplicate customer_ids in Customers_silver: 0


In [ ]:
products.head()

,product_id,product_name,category,price
0,1,Product_1,Books,4857
1,2,Product_2,Electronics,6290
2,3,Product_3,Fashion,2898
3,4,Product_4,Home,3529
4,5,Product_5,Books,4008


In [ ]:
products_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)



In [ ]:
products_silver.groupBy("product_id") \
.count() \
            .filter(col("count") > 1) \
            .count()

0

In [ ]:
orders_silver.groupBy("customer_id") \
            .count() \
            .filter(col("count") > 1) \
            .count()

99960

In [ ]:
products_silver.filter(
    col("price") <= 0
).count()

0

In [ ]:
orders_silver.filter(
    col("order_amount") <= 0
).count()

0

In [ ]:
invalid_customers = orders_silver.join(
    Customers_silver,
    "customer_id",
    "left_anti"
)

invalid_customers.count()

0

In [ ]:
valid_orders = orders_df.filter(
    col("order_amount") > 0
)

In [ ]:
valid_orders = orders_silver.filter(
    col("order_amount") > 0
)

clean_orders = valid_orders.join(
    Customers_silver,
    "customer_id",
    "inner"
)

In [ ]:
clean_orders = valid_orders

In [ ]:
clean_orders.write.mode("overwrite").parquet(
    "silver/orders"
)

In [ ]:
# Save to temporary storage before you leave
customers.to_parquet("customers_backup.parquet")


In [ ]:
bad_order = spark.createDataFrame([(9999,9999,101,-500.0,-2)]
                                  ,["order_id","customer_id","product_id","order_amount","quantity"])
orders_df = orders_df.union(bad_order)

In [ ]:
# Save to temporary storage before you leave
customers.to_parquet("customers_backup.parquet")


In [ ]:
# Resume exactly where you left off
import pandas as pd
customers = pd.read_parquet("customers_backup.parquet")


In [ ]:
orders_df.filter(col("order_amount") <=0).show()

+--------+-----------+----------+--------+------------+
|order_id|customer_id|product_id|quantity|order_amount|
+--------+-----------+----------+--------+------------+
|    9999|       9999|       101|  -500.0|          -2|
+--------+-----------+----------+--------+------------+



In [ ]:
#inserting bad_data
bad_order = spark.createDataFrame([
    (9998, 9999, 101, 500.0, 2)
], ["order_id", "customer_id", "product_id", "order_amount", "quantity"])

orders_df = orders_df.union(bad_order)

In [ ]:
#referential integriy
bad_order = spark.createDataFrame([
    (9998, 9999, 101, 500.0, 2)
], ["order_id", "customer_id", "product_id", "order_amount", "quantity"])

orders_df = orders_df.union(bad_order)

In [ ]:
from pyspark.sql.types import StringType, StructType, StructField, IntegerType

# Define the schema explicitly for duplicate_customer
duplicate_customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True)
])

#duplicate validation
duplicate_customer = spark.createDataFrame([
    (1, "Ram", "Bangalore", None)
], schema=duplicate_customer_schema)

customers_df = customers_df.union(duplicate_customer)

In [ ]:
# Save to temporary storage before you leave
customers.to_parquet("customers_backup.parquet")

In [ ]:
# Resume exactly where you left off
import pandas as pd
customers = pd.read_parquet("customers_backup.parquet")


In [ ]:
Customers_silver = customers_bronze_df.dropDuplicates(["customer_id"])

In [ ]:
products_silver = products_bronze_df.dropDuplicates(["product_id"])

In [ ]:
orders_silver = orders_bronze_df.dropDuplicates(["order_id"])

In [ ]:

# Remove records with null primary keys
from pyspark.sql.functions import col
Customers_silver=Customers_silver.filter(
  col("customer_id").isNotNull()
)


In [ ]:
# Remove records with null primary keys
products_silver=products_silver.filter(
    col("product_id").isNotNull()
)

In [ ]:
# Remove records with null primary keys
orders_silver = orders_silver.filter(
    col("order_id").isNotNull()
)

In [ ]:
#Busines rule validation
#price should be greater than zero
products_silver = products_silver.filter(col("price")>0)

In [ ]:
#orders quantity must be positive
orders_silver = orders_silver.filter(col("quantity")>0)

In [ ]:
#order amount must be positive
orders_silver =orders_silver.filter(col("order_amount")>0)

In [ ]:
#final validation
print("customers:",Customers_silver.count())
print("products :", products_silver.count())
print("orders :", orders_silver.count())

customers: 100000
products : 50000
orders : 1000000


In [ ]:

import os

# Define the base path for your project in Google Drive
base_path = "/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver"

# Create directories for customers, products, and orders if they don't exist
for folder in ["customers", "products", "orders"]:
    path = os.path.join(base_path, folder)
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"Created directory: {path}")
    else:
        print(f"Directory already exists: {path}")


Directory already exists: /content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/customers
Directory already exists: /content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/products
Directory already exists: /content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/orders


Now, let's save the silver layer DataFrames (`Customers_silver`, `products_silver`, `orders_silver`) to these newly created directories as Parquet files.

In [ ]:
Customers_silver.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/customers"
)
print("Customers_silver DataFrame saved.")

products_silver.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/products"
)
print("products_silver DataFrame saved.")

orders_silver.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/orders"
)
print("orders_silver DataFrame saved.")

Customers_silver DataFrame saved.
products_silver DataFrame saved.
orders_silver DataFrame saved.


The DataFrames have been saved. Now, the code in the next cell (which was previously failing) should be able to read them back. I will re-enable that cell to verify the read operation.

In [ ]:
#Read back the parquet files
Customers_parquet = spark.read.parquet(
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/customers"
)

products_parquet = spark.read.parquet(
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/products"
)

orders_parquet = spark.read.parquet(
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/orders"
)

### Verify Bronze Layer Directory Creation

In [ ]:
import os

bronze_base_path = "/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze"

# List contents of the Bronze base path
print(f"Contents of {bronze_base_path}:")
if os.path.exists(bronze_base_path):
    for item in os.listdir(bronze_base_path):
        print(item)
else:
    print(f"The directory {bronze_base_path} does not exist.")

Contents of /content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze:
customers.csv
products.csv
orders.csv
customers
products
orders


In [ ]:
!mkdir -p "/content/drive/MyDrive/Retail_Data_Engineering_Project"

!mkdir -p "/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze"

!mkdir -p "/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver"

!mkdir -p "/content/drive/MyDrive/Retail_Data_Engineering_Project/Gold"

!mkdir -p "/content/drive/MyDrive/Retail_Data_Engineering_Project/notebooks"

!mkdir -p "/content/drive/MyDrive/Retail_Data_Engineering_Project/sample_data"

In [ ]:
!mv "/content/drive/MyDrive/customers.csv" \
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze/"

!mv "/content/drive/MyDrive/products.csv" \
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze/"

!mv "/content/drive/MyDrive/orders.csv" \
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze/"

In [ ]:
!ls "/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze"

customers  customers.csv  orders  orders.csv  products	products.csv


In [ ]:
customers_df = spark.read.csv(
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Bronze/customers.csv",
header=True,
inferSchema=True
)

In [ ]:
Customers_silver.write.mode("overwrite").parquet(
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/customers"
)

products_silver.write.mode("overwrite").parquet(
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/products"
)

orders_silver.write.mode("overwrite").parquet(
"/content/drive/MyDrive/Retail_Data_Engineering_Project/Silver/orders"
)

In [ ]:
# GOLD LAYER STARTS HERE!
# Data will be transformed here to find:
# Customer Sales
#         ↓
# Product Sales
#         ↓
# State Sales
#         ↓
# Category Sales
#         ↓
# Top Customers

# Creating customer sales summary - 1. Join Orders with Customers

customer_orders = orders_silver.join(
    Customers_silver,
    "customer_id",
    "inner"
)

In [ ]:
#step 2 : Aggregate
#To find customer_sales
from pyspark.sql.functions import sum, avg, min, max, count
customer_sales = customer_orders.groupBy("customer_id","customer_name","state").agg(
    count("order_id").alias("total_orders"),
    sum("quantity").alias("total_quantity"),
    avg("order_amount").alias("total_revenue")
)

In [ ]:
customer_sales.show(10, truncate=False)

+-----------+--------------+-----------+------------+--------------+-----------------+
|customer_id|customer_name |state      |total_orders|total_quantity|total_revenue    |
+-----------+--------------+-----------+------------+--------------+-----------------+
|74411      |Customer_74411|Telangana  |10          |55            |3635.2           |
|66175      |Customer_66175|Kerala     |11          |57            |5114.818181818182|
|79620      |Customer_79620|Kerala     |12          |73            |4040.75          |
|78173      |Customer_78173|Maharashtra|8           |42            |4700.375         |
|63094      |Customer_63094|Kerala     |10          |56            |5426.6           |
|69633      |Customer_69633|Telangana  |10          |60            |4939.2           |
|64959      |Customer_64959|Tamil Nadu |18          |81            |6154.555555555556|
|34818      |Customer_34818|Tamil Nadu |9           |44            |4686.111111111111|
|17651      |Customer_17651|Telangana  |11 

In [ ]:
customer_sales.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Retail_Data_Engineering_Project/Gold/Customer_sales"
)

### Gold Layer: Product Sales

Next, let's calculate the total sales for each product. This involves joining the `customer_orders` DataFrame with `products_silver` to get product details and then aggregating by product information.

In [ ]:
#To find product_sales
product_sales = customer_orders.join(
    products_silver,
    "product_id",
    "inner"
).groupBy("product_id", "product_name", "category").agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("order_amount").alias("total_product_revenue")
)

product_sales.show(10, truncate=False)

+----------+-------------+-----------+-------------------+---------------------+
|product_id|product_name |category   |total_quantity_sold|total_product_revenue|
+----------+-------------+-----------+-------------------+---------------------+
|19473     |Product_19473|Sports     |95                 |97126                |
|25161     |Product_25161|Electronics|64                 |62593                |
|8060      |Product_8060 |Sports     |83                 |82706                |
|19060     |Product_19060|Electronics|50                 |63239                |
|31039     |Product_31039|Fashion    |88                 |96239                |
|6558      |Product_6558 |Electronics|80                 |82320                |
|11793     |Product_11793|Home       |134                |130147               |
|19924     |Product_19924|Books      |61                 |69933                |
|5469      |Product_5469 |Sports     |96                 |88625                |
|79        |Product_79   |El

In [ ]:
product_sales.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Retail_Data_Engineering_Project/Gold/Product_sales"
)
print("Product_sales DataFrame saved to Gold layer.")

Product_sales DataFrame saved to Gold layer.


### Gold Layer: State Sales

Now, let's calculate the total sales per state. This will provide insights into regional performance.

In [ ]:
#state_sales
state_sales = customer_orders.groupBy("state").agg(
    sum("order_amount").alias("total_state_revenue"),
    count("order_id").alias("total_orders_in_state")
)

state_sales.show(10, truncate=False)

+-----------+-------------------+---------------------+
|state      |total_state_revenue|total_orders_in_state|
+-----------+-------------------+---------------------+
|Karnataka  |1001672001         |198356               |
|Kerala     |1014032773         |200727               |
|Tamil Nadu |996699163          |197434               |
|Maharashtra|1023347033         |202892               |
|Telangana  |1013647696         |200591               |
+-----------+-------------------+---------------------+



In [ ]:
state_sales.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Retail_Data_Engineering_Project/Gold/State_sales"
)
print("State_sales DataFrame saved to Gold layer.")

State_sales DataFrame saved to Gold layer.


### Gold Layer: Category Sales

Next, we'll determine the total sales for each product category. This helps in understanding which product types are most popular.

In [ ]:
#category_sales
category_sales = customer_orders.join(
    products_silver,
    "product_id",
    "inner"
).groupBy("category").agg(
    sum("order_amount").alias("total_category_revenue"),
    sum("quantity").alias("total_category_quantity")
)

category_sales.show(10, truncate=False)

+-----------+----------------------+-----------------------+
|category   |total_category_revenue|total_category_quantity|
+-----------+----------------------+-----------------------+
|Home       |1014696977            |1006187                |
|Fashion    |998890330             |991234                 |
|Sports     |1010251493            |996795                 |
|Electronics|1000957378            |991780                 |
|Books      |1024602488            |1014571                |
+-----------+----------------------+-----------------------+



In [ ]:
category_sales.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Retail_Data_Engineering_Project/Gold/Category_sales"
)
print("Category_sales DataFrame saved to Gold layer.")

Category_sales DataFrame saved to Gold layer.


### Gold Layer: Top Customers

Finally, let's identify the top customers based on their total revenue. We can use the already created `customer_sales` DataFrame for this.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc

# Order customers by total_revenue in descending order to find top customers
top_customers = customer_sales.orderBy(desc("total_revenue"))

top_customers.show(10, truncate=False)

+-----------+--------------+-----------+------------+--------------+-------------+
|customer_id|customer_name |state      |total_orders|total_quantity|total_revenue|
+-----------+--------------+-----------+------------+--------------+-------------+
|89505      |Customer_89505|Telangana  |1           |2             |9907.0       |
|45538      |Customer_45538|Maharashtra|1           |3             |9471.0       |
|93393      |Customer_93393|Tamil Nadu |2           |6             |9380.0       |
|26295      |Customer_26295|Kerala     |3           |14            |9249.0       |
|5903       |Customer_5903 |Telangana  |4           |20            |9185.0       |
|99457      |Customer_99457|Kerala     |2           |16            |9140.0       |
|42804      |Customer_42804|Maharashtra|2           |14            |9135.0       |
|5771       |Customer_5771 |Telangana  |2           |7             |9132.0       |
|64061      |Customer_64061|Karnataka  |4           |11            |9107.0       |
|601

In [ ]:
top_customers.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Retail_Data_Engineering_Project/Gold/Top_customers"
)
print("Top_customers DataFrame saved to Gold layer.")

Top_customers DataFrame saved to Gold layer.


In [ ]:
#Then why use Window Functions?

#Window functions become useful when the requirement changes.

#Suppose the business asks:

##Find the Top 3 customers in each state.

#Now, orderBy() alone cannot do this.

#You need to rank customers within each state.
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.window import Window

window = Window.partitionBy("state").orderBy(desc("total_revenue"))

top_customers = customer_sales.withColumn(
    "rank",
    dense_rank().over(window)
).filter("rank <= 3")

In [ ]:
top_customers.show(20, truncate=False)

+-----------+--------------+-----------+------------+--------------+-------------+----+
|customer_id|customer_name |state      |total_orders|total_quantity|total_revenue|rank|
+-----------+--------------+-----------+------------+--------------+-------------+----+
|64061      |Customer_64061|Karnataka  |4           |11            |9107.0       |1   |
|60809      |Customer_60809|Karnataka  |2           |13            |9057.5       |2   |
|1108       |Customer_1108 |Karnataka  |4           |23            |8991.0       |3   |
|26295      |Customer_26295|Kerala     |3           |14            |9249.0       |1   |
|99457      |Customer_99457|Kerala     |2           |16            |9140.0       |2   |
|60189      |Customer_60189|Kerala     |2           |11            |9088.5       |3   |
|45538      |Customer_45538|Maharashtra|1           |3             |9471.0       |1   |
|42804      |Customer_42804|Maharashtra|2           |14            |9135.0       |2   |
|15084      |Customer_15084|Maha

## Pausing and Resuming Work: Ensuring Smooth Continuity

When you need to pause your work and return later, it's essential to ensure that your processed data is persistently stored and easily reloadable. We've already saved your Gold layer DataFrames as Parquet files in Google Drive, which is the best practice for persistence.

To resume your work smoothly, you typically need to perform the following steps:

1.  **Remount Google Drive:** If your Colab session has disconnected or restarted, your Google Drive will be unmounted. You'll need to re-run the `drive.mount()` command.
2.  **Re-initialize Spark Session:** The Spark session is tied to the Colab runtime. If the runtime restarts, you'll need to re-create your SparkSession.
3.  **Reload DataFrames:** Instead of reprocessing all layers, you can directly load your Gold layer DataFrames from their saved Parquet locations. This allows you to quickly continue your analysis.

Let's add the necessary cells to demonstrate this.

In [ ]:
# Step 1: Remount Google Drive (if necessary)
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Re-create Spark Session (if kernel restarted)
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Retail Data Engineering Project") \
    .getOrCreate()

print("Spark Session Re-Created Successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark Session Re-Created Successfully


### Step 3: Reloading Gold Layer DataFrames

Now, you can directly load your Gold layer DataFrames from their Parquet files. This allows you to immediately access the results of your data pipeline without re-running all previous steps.

In [ ]:
gold_base_path = "/content/drive/MyDrive/Retail_Data_Engineering_Project/Gold"

reloaded_customer_sales = spark.read.parquet(f"{gold_base_path}/Customer_sales")
reloaded_product_sales = spark.read.parquet(f"{gold_base_path}/Product_sales")
reloaded_state_sales = spark.read.parquet(f"{gold_base_path}/State_sales")
reloaded_category_sales = spark.read.parquet(f"{gold_base_path}/Category_sales")
reloaded_top_customers = spark.read.parquet(f"{gold_base_path}/Top_customers")

print("Gold layer DataFrames reloaded successfully.")

# You can display a sample to verify
reloaded_customer_sales.show(5, truncate=False)
reloaded_top_customers.show(5, truncate=False)

Gold layer DataFrames reloaded successfully.
+-----------+--------------+----------+------------+--------------+-----------------+
|customer_id|customer_name |state     |total_orders|total_quantity|total_revenue    |
+-----------+--------------+----------+------------+--------------+-----------------+
|54467      |Customer_54467|Kerala    |11          |50            |3829.090909090909|
|92619      |Customer_92619|Tamil Nadu|9           |35            |5002.111111111111|
|44617      |Customer_44617|Karnataka |10          |50            |5704.7           |
|72920      |Customer_72920|Kerala    |6           |26            |5992.666666666667|
|60097      |Customer_60097|Kerala    |6           |34            |4490.333333333333|
+-----------+--------------+----------+------------+--------------+-----------------+
only showing top 5 rows
+-----------+--------------+----------+------------+--------------+-----------------+
|customer_id|customer_name |state     |total_orders|total_quantity|tota

In [ ]:
Window.partitionBy("state").orderBy(desc("total_revenue"))

In [ ]:
gold_base_path = "/content/drive/MyDrive/Retail_Data_Engineering_Project/Gold"

reloaded_customer_sales = spark.read.parquet(f"{gold_base_path}/Customer_sales")
reloaded_product_sales = spark.read.parquet(f"{gold_base_path}/Product_sales")
reloaded_state_sales = spark.read.parquet(f"{gold_base_path}/State_sales")
reloaded_category_sales = spark.read.parquet(f"{gold_base_path}/Category_sales")
reloaded_top_customers = spark.read.parquet(f"{gold_base_path}/Top_customers")

print("Gold layer DataFrames reloaded successfully.")

# You can display a sample to verify
reloaded_customer_sales.show(5, truncate=False)
reloaded_top_customers.show(5, truncate=False)

Gold layer DataFrames reloaded successfully.
+-----------+--------------+----------+------------+--------------+-----------------+
|customer_id|customer_name |state     |total_orders|total_quantity|total_revenue    |
+-----------+--------------+----------+------------+--------------+-----------------+
|54467      |Customer_54467|Kerala    |11          |50            |3829.090909090909|
|92619      |Customer_92619|Tamil Nadu|9           |35            |5002.111111111111|
|44617      |Customer_44617|Karnataka |10          |50            |5704.7           |
|72920      |Customer_72920|Kerala    |6           |26            |5992.666666666667|
|60097      |Customer_60097|Kerala    |6           |34            |4490.333333333333|
+-----------+--------------+----------+------------+--------------+-----------------+
only showing top 5 rows
+-----------+--------------+----------+------------+--------------+-----------------+
|customer_id|customer_name |state     |total_orders|total_quantity|tota

In [ ]:
#finding top 3 customers
top3 = top_customers.filter("rank <= 3")


## Summary

This notebook demonstrates a comprehensive data engineering pipeline, moving data through Bronze, Silver, and Gold layers.

*   **Bronze Layer:** Raw data from CSVs (`customers`, `products`, `orders`) is ingested and stored as Parquet files, ensuring data immutability and enabling re-processing.
*   **Silver Layer:** Data from the Bronze layer undergoes quality checks, including schema validation, null value checks, duplicate removal, and business rule validations (e.g., positive prices, quantities, and order amounts). The cleaned and validated data is then saved as Parquet files.
*   **Gold Layer:** Transformed and aggregated data is prepared for analytical consumption. This includes calculating customer sales, product sales, state sales, category sales, and identifying top customers using Spark SQL window functions. The final aggregated data is also stored as Parquet for easy access and analysis.